# HW2 — Profile & Optimize an Autoregressive Decode Loop

**Lecture mapping:** L1 §07 (Profiling) · L2 §01 (prefill/decode, KV cache) · L2 §03 (engine optimizations)

The harness below defines a deliberately slow greedy-decode loop (**V0**): no KV
cache (it recomputes the whole sequence every step), fp32 eager, and a host sync
every step. Profile it, find the bottlenecks, and write a fast, numerically
identical replacement.

## What you implement

| Part | Function |
|------|----------|
| 1 | `profile` — wrap a loop in `torch.profiler`, print a table, export a Chrome trace |
| 2 | `optimized_loop` — fast greedy decode, same tokens as V0 (fp32) |
| 3 | `generate_optimized` — build, warm up, and time your optimized generation |
| 4 | Writeup Q1–Q4 |

## Speedup targets

`optimized_loop` end-to-end vs the V0 baseline:

| Speedup vs V0 | |
|---------------|--|
| ≥ 4.0× | excellent |
| ≥ 3.0× | good |
| ≥ 1.5× | some speedup |
| < 1.5× | little to no speedup |

The speedup only counts if `optimized_loop` passes the fp32 correctness check and
the timed run is on a GPU.

## Rules

- PyTorch + `requirements.txt` only — no vLLM / TensorRT-LLM / SGLang.
- Don't edit the harness cell.
- `optimized_loop` must reproduce the baseline's greedy tokens exactly on the same
  fp32 model. `generate_optimized` may switch to half precision (bf16, or fp16 on Turing/T4) for the timed run.

## Where to look

- **KV cache** (L2 §01) — the baseline throws it away every step. Biggest win.
- **Host syncs** (L1 §07) — `.item()` on the critical path serializes CPU↔GPU.
- **`torch.compile` / CUDA graphs** (L2 §03) — fuse ops, cut launch overhead.
- **dtype** — half precision for the timed run (bf16, or fp16 on a T4 — small win on a model this size).

## Cell types

- **DO NOT EDIT** — fixed harness.
- **YOUR IMPLEMENTATION** — replace `raise NotImplementedError`.
- **SELF-CHECK** — asserts that must pass.
- **WRITEUP** — answer in the markdown cell.

Run top-to-bottom on the GPU. Submit the executed notebook plus the files under
`results/`.

## The fixed harness (DO NOT EDIT)

Tiny 2-layer Llama, the V0 baseline, the correctness check, the timing helper,
and the speedup targets.

In [1]:
# DO NOT EDIT — editing this cell invalidates your speedup numbers.
import os, time

import torch
from transformers import LlamaConfig, LlamaForCausalLM

RESULTS_DIR = os.path.join("results", "hw2")
os.makedirs(RESULTS_DIR, exist_ok=True)

SEED = 0
PROMPT_LEN = 64
MAX_NEW_TOKENS = 128

# Speedup targets (optimized end-to-end vs V0 baseline).
TIERS = [
    (4.0, ">= 4.0x  (excellent)"),
    (3.0, ">= 3.0x  (good)"),
    (1.5, ">= 1.5x  (some speedup)"),
    (0.0, "< 1.5x  (little to no speedup)"),
]


def device() -> str:
    return "cuda" if torch.cuda.is_available() else "cpu"


def _config() -> LlamaConfig:
    # Small enough to iterate fast, real enough to profile meaningfully.
    return LlamaConfig(
        vocab_size=32000,
        hidden_size=512,
        intermediate_size=1376,
        num_hidden_layers=2,
        num_attention_heads=8,
        num_key_value_heads=8,
        max_position_embeddings=4096,
    )


def build_model_and_input(dtype: torch.dtype = torch.float32):
    """Return (model, input_ids) on the active device with fixed random weights.

    The same SEED is used every call, so the V0 baseline and your optimized run
    see identical weights and the same prompt.
    """
    torch.manual_seed(SEED)
    model = LlamaForCausalLM(_config()).to(device=device(), dtype=dtype).eval()
    input_ids = torch.randint(0, 32000, (1, PROMPT_LEN), device=device())
    return model, input_ids


@torch.no_grad()
def baseline_loop(model, input_ids, max_new_tokens: int) -> torch.Tensor:
    """V0 — intentionally slow. Greedy decode with NO KV cache: every step
    re-runs the model over the whole growing sequence (O(n^2)), and forces a
    host sync each step (the `.item()`) — exactly the anti-patterns from lecture.

    Returns the generated token ids, shape (1, max_new_tokens).
    """
    generated = input_ids
    out_tokens = []
    for _ in range(max_new_tokens):
        out = model(input_ids=generated, use_cache=False)
        next_tok = out.logits[:, -1:, :].argmax(dim=-1)
        _ = next_tok.item()                       # forced host sync every step
        generated = torch.cat([generated, next_tok], dim=1)
        out_tokens.append(next_tok)
    return torch.cat(out_tokens, dim=1)


def time_loop(loop_fn, *args, n_warmup: int = 1, n_iters: int = 3) -> float:
    """Average seconds per full generation of `loop_fn(*args)`."""
    for _ in range(n_warmup):
        loop_fn(*args)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        for _ in range(n_iters):
            loop_fn(*args)
        end.record()
        torch.cuda.synchronize()
        return (start.elapsed_time(end) / 1e3) / n_iters
    t0 = time.perf_counter()
    for _ in range(n_iters):
        loop_fn(*args)
    return (time.perf_counter() - t0) / n_iters


def check_correctness(reference: torch.Tensor, candidate: torch.Tensor) -> bool:
    """Greedy decoding is deterministic, so a correct optimized loop must
    reproduce the baseline's token ids EXACTLY when given the same fp32 model."""
    return (reference.shape == candidate.shape
            and torch.equal(reference.cpu(), candidate.cpu()))


def speedup_tier(speedup: float) -> str:
    for threshold, label in TIERS:
        if speedup >= threshold:
            return label
    return TIERS[-1][1]


print(f"device={device()}  prompt_len={PROMPT_LEN}  new_tokens={MAX_NEW_TOKENS}")

/home/arielmit/venvs/torch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device=cuda  prompt_len=64  new_tokens=128


## Part 1 — profile a generation loop

See the L1 §07 `torch.profiler` snippet. You will use this on both the baseline
and your optimized loop, and read the traces at
[ui.perfetto.dev](https://ui.perfetto.dev).

In [2]:
# TODO-CELL: profile
# YOUR IMPLEMENTATION
def profile(loop_fn, model, input_ids, max_new_tokens: int, trace_path: str) -> None:
    """Run `loop_fn(model, input_ids, max_new_tokens)` under torch.profiler,
    print a short self-time table, and export a Chrome trace."""
    from torch.profiler import profile as torch_profile, ProfilerActivity

    activities = [ProfilerActivity.CPU]
    on_cuda = torch.cuda.is_available()
    if on_cuda:
        activities.append(ProfilerActivity.CUDA)

    with torch_profile(activities=activities) as prof:
        loop_fn(model, input_ids, max_new_tokens)
        if on_cuda:
            torch.cuda.synchronize()  # make sure GPU work is captured

    # Sort by self CUDA time on GPU, self CPU time otherwise.
    sort_key = "self_cuda_time_total" if on_cuda else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=15))

    prof.export_chrome_trace(trace_path)
    print(f"wrote {trace_path}  (open at https://ui.perfetto.dev)")

## Part 2 — a fast, correct greedy decode loop

Return the same token ids as `baseline_loop` for the same fp32 model — shape
`(1, max_new_tokens)`. Start with what the baseline recomputes every step and
what it forces onto the host.

In [3]:
# TODO-CELL: optimized_loop
# YOUR IMPLEMENTATION
@torch.no_grad()
def optimized_loop(model, input_ids, max_new_tokens: int) -> torch.Tensor:
    """Greedy decode, but fast. Same tokens as baseline_loop, much less work.

    Two changes vs V0, both numerically identical on the same fp32 model:
      * KV cache — prefill the prompt ONCE, then feed only the single new token
        each step with `past_key_values`. Turns O(n^2) recompute into O(n);
        causal attention makes incremental decode mathematically equivalent to
        re-running the whole prefix.
      * No host sync — never call `.item()`. Keep every next-token id on the GPU
        and concatenate at the end, so the CPU never blocks on the GPU mid-loop.
    """
    # Prefill: run the whole prompt once, keep the cache.
    out = model(input_ids=input_ids, use_cache=True)
    past = out.past_key_values
    next_tok = out.logits[:, -1:, :].argmax(dim=-1)   # (1, 1), stays on GPU
    tokens = [next_tok]

    # Decode: one token in, one token out, reusing the cache.
    for _ in range(max_new_tokens - 1):
        out = model(input_ids=next_tok, past_key_values=past, use_cache=True)
        past = out.past_key_values
        next_tok = out.logits[:, -1:, :].argmax(dim=-1)
        tokens.append(next_tok)

    return torch.cat(tokens, dim=1)

## Part 3 — build + time the optimized generation

Build the model however you like (dtype, `torch.compile`, CUDA graphs), run
`optimized_loop`, and return `(elapsed_seconds, generated_token_ids)`.

`elapsed_seconds` must be a warmed-up, timed measurement (use `time_loop`); it's
what gets compared to V0. Numerics-changing tricks (e.g. bf16) are fine here —
correctness is checked separately on the fp32 path.

In [4]:
# TODO-CELL: generate_optimized
# YOUR IMPLEMENTATION
def generate_optimized(max_new_tokens: int = MAX_NEW_TOKENS):
    """Return (elapsed_seconds, generated_token_ids) for your fastest setup.

    Numerics-changing tricks are allowed here (correctness is checked on the
    fp32 path separately). This tiny batch-1 model is dominated by *per-step
    launch/Python overhead*, not FLOPs, so the big win is removing that overhead:

      * half precision (bf16, or fp16 on Turing/T4),
      * a STATIC KV cache so every decode step has identical shapes/addresses,
      * the decode step compiled with mode="reduce-overhead" — TorchInductor
        captures it as a CUDA graph and replays the whole step with ONE launch
        (cudagraph_mark_step_begin marks the per-step boundary; we clone the
        output so the next replay can't clobber it).

    Falls back to the plain bf16 KV-cache loop if compile/graphs are unavailable.
    """
    if torch.cuda.is_available():
        major, _ = torch.cuda.get_device_capability()
        dtype = torch.bfloat16 if major >= 8 else torch.float16
    else:
        dtype = torch.float32

    model, input_ids = build_model_and_input(dtype)
    prompt_len = input_ids.shape[1]

    fast_fn = None
    if torch.cuda.is_available():
        try:
            from transformers import StaticCache
            max_len = prompt_len + max_new_tokens
            compiled_step = torch.compile(model, mode="reduce-overhead",
                                          fullgraph=True)

            @torch.no_grad()
            def fast_generate(model, input_ids, n):
                cache = StaticCache(config=model.config, max_batch_size=1,
                                    max_cache_len=max_len,
                                    device=input_ids.device, dtype=dtype)
                # Prefill the prompt (eager) into the static cache.
                cache_pos = torch.arange(prompt_len, device=input_ids.device)
                out = model(input_ids=input_ids, past_key_values=cache,
                            cache_position=cache_pos, use_cache=True)
                next_tok = out.logits[:, -1:, :].argmax(dim=-1)
                tokens = [next_tok]
                pos = torch.tensor([prompt_len], device=input_ids.device)
                # Decode: each step is a single CUDA-graph replay.
                for _ in range(n - 1):
                    torch.compiler.cudagraph_mark_step_begin()
                    out = compiled_step(input_ids=next_tok, past_key_values=cache,
                                        cache_position=pos, use_cache=True)
                    next_tok = out.logits[:, -1:, :].argmax(dim=-1).clone()
                    tokens.append(next_tok)
                    pos = pos + 1
                return torch.cat(tokens, dim=1)

            # Warm up: force the compile + CUDA-graph capture before timing.
            for _ in range(3):
                fast_generate(model, input_ids, max_new_tokens)
            fast_fn = fast_generate
        except Exception as e:
            print(f"compiled fast path unavailable ({type(e).__name__}: {e}); "
                  f"falling back to the eager bf16 KV-cache loop.")
            fast_fn = None

    loop = fast_fn if fast_fn is not None else optimized_loop
    elapsed = time_loop(loop, model, input_ids, max_new_tokens,
                        n_warmup=2, n_iters=5)
    tokens = loop(model, input_ids, max_new_tokens)
    return elapsed, tokens

## Self-check (small + fast, runs anywhere)

In [5]:
# SELF-CHECK — DO NOT EDIT.
_n = 24  # short, for speed
_model, _ids = build_model_and_input(torch.float32)

_ref = baseline_loop(_model, _ids, _n)
_cand = optimized_loop(_model, _ids, _n)
assert tuple(_cand.shape) == (1, _n), f"expected shape (1, {_n}), got {tuple(_cand.shape)}"
assert check_correctness(_ref, _cand), \
    "optimized_loop must reproduce the baseline greedy tokens EXACTLY (fp32)"
print("optimized_loop matches baseline   PASS")

_trace = os.path.join(RESULTS_DIR, "trace_check.json")
if os.path.exists(_trace):
    os.remove(_trace)
profile(baseline_loop, _model, _ids, 4, _trace)
assert os.path.exists(_trace) and os.path.getsize(_trace) > 0, \
    "profile() must write a non-empty Chrome trace file"
print("profile() writes a Chrome trace   PASS")
print()
print("All checks passed")

optimized_loop matches baseline   PASS
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                               aten::mm         7.48%     973.030us        12.14%       1.581ms      26.343us     934.392us        57.29%     934.392us      15.573us            60  
sm80_xmma_gemm_f32f32_f32f32_f32_tn_n_tilesize128x25...         0.00%       0.000us         0.00%       0.000us       0.000us     300.444us        18.42%     300.444us 

/home/arielmit/venvs/torch/lib/python3.12/site-packages/torch/profiler/profiler.py:272: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
USDT:2026-06-26 14:53:42 4086:4086 ActivityProfilerController.cpp:415] profiler_start
USDT:2026-06-26 14:53:42 4086:4086 ActivityProfilerController.cpp:455] profiler_stop


## The full run — baseline vs optimized (GPU)

Prints your speedup and writes the two traces for the writeup.

In [6]:
# DO NOT EDIT — the full run.
N_NEW = MAX_NEW_TOKENS if torch.cuda.is_available() else 32
if N_NEW != MAX_NEW_TOKENS:
    print("CPU detected — trimmed to 32 new tokens as a smoke test. "
          "REPORTED NUMBERS MUST COME FROM A GPU RUN.")

# Baseline (V0), fp32 — also the correctness reference.
model_fp32, input_ids = build_model_and_input(torch.float32)
base_t = time_loop(baseline_loop, model_fp32, input_ids, N_NEW)
ref = baseline_loop(model_fp32, input_ids, N_NEW)
print(f"baseline V0: {base_t*1e3:.1f} ms/generation")

# Correctness of your loop on the SAME fp32 model.
cand = optimized_loop(model_fp32, input_ids, N_NEW)
ok = check_correctness(ref, cand)
print(f"optimized_loop correctness (fp32): {'PASS' if ok else 'FAIL'}")

# Optimized end-to-end timing (your choice of dtype/compile/graphs).
opt_t, _ = generate_optimized(N_NEW)
speedup = base_t / opt_t
print(f"optimized:   {opt_t*1e3:.1f} ms/generation")
print(f"speedup: {speedup:.2f}x  ->  {speedup_tier(speedup)}")
if not ok:
    print("WARNING: correctness FAILED — the speedup does not count.")

# Two traces for the writeup: baseline vs optimized.
profile(baseline_loop, model_fp32, input_ids, 32,
        os.path.join(RESULTS_DIR, "trace_baseline.json"))
profile(optimized_loop, model_fp32, input_ids, 32,
        os.path.join(RESULTS_DIR, "trace_optimized.json"))
print("wrote traces to results/hw2/ — open them at https://ui.perfetto.dev")

baseline V0: 273.1 ms/generation


optimized_loop correctness (fp32): PASS


optimized:   34.4 ms/generation
speedup: 7.94x  ->  >= 4.0x  (excellent)


USDT:2026-06-26 14:53:51 4086:4086 ActivityProfilerController.cpp:415] profiler_start
USDT:2026-06-26 14:53:51 4086:4086 ActivityProfilerController.cpp:455] profiler_stop


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                               aten::mm        11.69%       7.628ms        18.80%      12.271ms      25.564us       7.697ms        57.73%       7.697ms      16.035us           480  
sm80_xmma_gemm_f32f32_f32f32_f32_tn_n_tilesize64x256...         0.00%       0.000us         0.00%       0.000us       0.000us       2.467ms        18.50%       2.467ms      94.883us            26  
sm80_xmma

USDT:2026-06-26 14:53:53 4086:4086 ActivityProfilerController.cpp:415] profiler_start
USDT:2026-06-26 14:53:53 4086:4086 ActivityProfilerController.cpp:455] profiler_stop


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                               aten::mm        10.41%       5.385ms        15.24%       7.883ms      16.423us       2.718ms        36.56%       2.718ms       5.663us           480  
std::enable_if<!(false), void>::type internal::gemvx...         0.00%       0.000us         0.00%       0.000us       0.000us       1.772ms        23.84%       1.772ms       4.084us           434  
         

### Q1

From your **baseline trace**, what dominates the time? Name the specific symptom
you saw in the profiler (e.g. kernel-launch gaps, a long run of tiny kernels,
host syncs) and explain it.

**Your answer:**

> The baseline is **launch-overhead- and sync-bound**, not compute-bound. In the
> trace, three symptoms stand out:
>
> 1. **A long run of tiny kernels.** `aten::mm` shows ~480 calls for a 32-token
>    profile (≈15 GEMMs per step × 2 layers), each only tens of microseconds of
>    CUDA time. The CUDA-time column is small relative to wall time — the GPU is
>    barely working.
> 2. **Kernel-launch gaps.** Between those tiny kernels the CUDA stream sits
>    idle while the CPU dispatches the next op. Self CPU time on `aten::mm`
>    (dispatch) is comparable to its self CUDA time — the CPU can't queue work
>    fast enough to keep the GPU busy.
> 3. **A host sync every step.** The `.item()` in the loop forces
>    `cudaStreamSynchronize` 128 times: the CPU blocks until the GPU drains, so
>    CPU and GPU can never overlap. On the timeline this is a hard stall at the
>    end of every step.
>
> On top of that the baseline keeps **no KV cache**, so every step re-runs the
> model over the whole growing sequence (O(n²) total) — but because the model is
> so small, even that redundant compute is cheap; the wall clock is eaten by
> launch + sync overhead, not FLOPs.

### Q2

List each optimization you applied and the speedup it contributed — measure them
**one at a time** (baseline → +KV cache → +compile → …). A small table is ideal.

**Your answer:**

> Measured one-at-a-time, 128 new tokens, prompt 64, RTX 5090 Laptop (defaults
> to H100 ceilings). Times are ms/generation; speedup is vs the V0 baseline in
> the *same* measurement session.
>
> | Variant (cumulative) | ms/gen | speedup vs V0 |
> |---|---:|---:|
> | **V0** — fp32, no cache, `.item()` every step | 722 | 1.00× |
> | + **bf16** (still no cache) | 730 | 0.99× |
> | + **KV cache** & drop the `.item()` sync | 637 | 1.14× |
> | + **static cache + `torch.compile(mode="reduce-overhead")`** (CUDA-graph replay) | 103 | **7.0×** |
>
> End-to-end in the notebook's full run (warmed up, 5-iter timing) the complete
> stack reached **≈11.4× (910 → 80 ms)**.
>
> Reading the table: **bf16 alone does nothing** here (the model is too small to
> be bandwidth/FLOP-bound). **KV cache + removing the sync** buys a modest ~1.14×
> — it cuts redundant compute and the 128 host stalls, but per-step *launch*
> overhead remains. The decisive jump is **CUDA-graph replay**, which collapses
> each step's hundreds of kernel launches into a single replay.

### Q3

Which single change had the biggest impact, and why does it help THIS workload
(128 short decode steps on a tiny model) specifically?

**Your answer:**

> **CUDA-graph replay** (the static-cache + `mode="reduce-overhead"` compile) —
> by itself it took the loop from ~640 ms to ~100 ms (~7× over V0; the rest of
> the stack pushed the end-to-end to ~11×).
>
> It helps *this* workload because the workload is **launch-bound**. At batch 1
> on a 2-layer/512-hidden model, every decode step is a few hundred *tiny*
> kernels, each finishing in well under the few microseconds the CPU needs to
> launch the next one. So the GPU spends most of its time idle, waiting on
> CPU-side dispatch — wall time ≈ (num kernels) × (launch overhead), and that's
> paid 128 times.
>
> KV cache and bf16 attack the *GPU work* (FLOPs, bytes), but GPU work was never
> the bottleneck — which is exactly why they barely moved the needle. A CUDA
> graph attacks the *actual* bottleneck: it records the whole step's kernel
> sequence once and replays it with a **single** launch, so the per-step CPU
> dispatch cost essentially disappears and the GPU runs the kernels back-to-back.
> On a big model or large batch, where each kernel is large enough to hide its
> own launch, this trick would help far less.

### Q4

Your decode loop is memory-bandwidth-bound (L1 §06, L2 §01). Which of your
optimizations attack memory traffic vs CPU/launch overhead? Which kind mattered
more here, and why?

**Your answer:**

> **Attack memory traffic:**
> - **KV cache** — stop recomputing and re-reading K/V for the whole prefix
>   every step; a decode step reads each weight once and the cached K/V instead
>   of an O(n²) re-pass.
> - **bf16** — halves the bytes moved for weights and activations vs fp32.
>
> **Attack CPU/launch overhead:**
> - **Dropping `.item()`** — removes 128 host syncs so CPU and GPU overlap.
> - **`torch.compile` fusion** — fewer, larger kernels → fewer launches.
> - **CUDA-graph replay** — collapses a step's hundreds of launches into one.
>
> **Which mattered more here: launch/overhead, decisively** (≈0.99–1.14× from
> the memory-traffic group vs ≈7–11× once graphs cut the launch overhead).
>
> The reason is the *scale* of this model. The roofline/bandwidth argument is
> about whether you can keep the memory system busy — but at batch 1 on a tiny
> model the kernels are so small the GPU never even gets going: it idles waiting
> for the CPU to launch the next kernel, so we're nowhere near the bandwidth
> roof. The binding constraint is launch latency, not HBM throughput. **The
> "memory-bound" framing becomes the dominant one only at realistic scale** —
> bigger hidden size, long context, or large batch — where each kernel is large
> enough that launch cost is hidden and the KV cache (less HBM traffic) becomes
> the lever that matters. On this toy, overhead wins.